
# Inspect Batches Without Training

生成一次模拟数据（simulate 模式，episode>0），加载可选 ckpt（ep*_*.pt），构造各模块的 batches，并查看形状/示例。


In [1]:

import sys
from pathlib import Path
import torch
import pandas as pd

# 路径设置：假设 notebook 位于 experiments 目录下
ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

from config import Config
from training.episode import Episode, HyperParams
from data.simulate_ts import SimulateTS
from experiments import run_utils as utils


In [9]:

# 可选：设置 ckpt 目录（含 ep*_*.pt），留空则用随机初始化
ckpt_dir = None  # e.g., ROOT.parent / "cachedir/20250305_1530/checkpoints"

# 小规模配置以快速运行
n_paths = 3
horizon = 10
group_size = 100
batch_size = 8
n_branches = Config.BRANCH_NUM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Config.DEVICE = device

models = utils.build_models(device)
if ckpt_dir:
    ckpt_dir = Path(ckpt_dir)
    for name in ["sdf_fc1", "policy_value", "fc2"]:
        p = ckpt_dir / f"ep0_{name}.pt"
        if p.exists():
            state = torch.load(p, map_location=device)
            models[name].load_state_dict(state, strict=False)
            print(f"Loaded {name} from {p}")
        else:
            print(f"[warn] missing {p}, using random init for {name}")

hp = HyperParams()
hp.n_paths = n_paths
hp.simulate_horizon = horizon


In [10]:

# 生成一次模拟数据
dsim = SimulateTS(
    models=models,
    config=Config,
    n_paths=n_paths,
    group_size=group_size,
    branch_num=n_branches,
    horizon=horizon,
    device=device,
)
df_firm, df_macro = dsim.simulate()
print("df_firm shape:", df_firm.shape)
print(df_firm.head())
print("df_macro shape:", df_macro.shape)
print(df_macro.head())


Simulating paths: 100%|██████████| 3/3 [00:00<00:00,  4.32it/s]

df_firm shape: (13050, 27)
   path  t  branch                                ID  entry         b  \
0     0  0      -1  6c735502c0ab4839b5f630b9f5dd7909    1.0  0.370202   
1     0  0      -1  1012c97ea81245db914f5d6df9ec3f69    1.0  0.003481   
2     0  0      -1  d820697891ec4cd7b9d0395c098e1be4    1.0  0.261684   
3     0  0      -1  67ae1926f8a94333929cbc3b74c87c11    1.0  0.225235   
4     0  0      -1  abdc61e8877c45439ed4dfaaca154a76    1.0  0.056402   

          z  ETA         i        x  ...     Bar_i         Bar_z         P  \
0  0.090160  1.0  0.126619 -0.01342  ...  0.349796  4.665106e-17  0.752077   
1 -0.310053  0.0  0.193134 -0.01342  ...  0.354465  5.421784e-17  0.749070   
2 -0.553397  0.0  0.141070 -0.01342  ...  0.365960  6.926503e-17  0.744172   
3 -0.294209  0.0  0.029609 -0.01342  ...  0.358588  6.910140e-17  0.744219   
4  0.016489  0.0  0.078568 -0.01342  ...  0.347262  4.888019e-17  0.751143   

        bp0       bpI        bp         Y         I           Phi

In [15]:
df_firm[df_firm['path'] == 0]

,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_i,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C
0,0,0,-1,6c735502c0ab4839b5f630b9f5dd7909,1.0,0.370202,0.090160,1.0,0.126619,-0.013420,...,0.349796,4.665106e-17,0.752077,0.552019,0.484954,0.528560,1.592163,0.212764,7.153271e-17,1.379399
1,0,0,-1,1012c97ea81245db914f5d6df9ec3f69,1.0,0.003481,-0.310053,0.0,0.193134,-0.013420,...,0.354465,5.421784e-17,0.749070,0.554078,0.476500,0.526579,0.908869,0.211582,5.868682e-17,0.697287
2,0,0,-1,d820697891ec4cd7b9d0395c098e1be4,1.0,0.261684,-0.553397,0.0,0.141070,-0.013420,...,0.365960,6.926503e-17,0.744172,0.554633,0.472643,0.524628,0.470990,0.125878,4.506307e-17,0.345111
3,0,0,-1,67ae1926f8a94333929cbc3b74c87c11,1.0,0.225235,-0.294209,0.0,0.029609,-0.013420,...,0.358588,6.910140e-17,0.744219,0.552662,0.470195,0.523090,0.392548,0.059063,3.201088e-17,0.333485
4,0,0,-1,abdc61e8877c45439ed4dfaaca154a76,1.0,0.056402,0.016489,0.0,0.078568,-0.013420,...,0.347262,4.888019e-17,0.751143,0.551830,0.473917,0.524774,1.389766,0.176352,6.782794e-17,1.213413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4345,0,10,1,13473e2cebe9490ea091dbf4c4557246,0.0,0.631631,-0.587846,1.0,0.038991,0.059482,...,0.456923,1.030618e-15,0.690172,0.523878,0.492681,0.509623,0.592186,0.118339,8.227559e-16,0.473847
4346,0,10,1,9e4498b5c07f48608f3a62b291f7ec99,0.0,0.625130,-0.143623,1.0,0.022294,0.059482,...,0.459481,1.073871e-15,0.689350,0.520437,0.489051,0.506015,0.923679,0.110769,1.035448e-15,0.812910
4347,0,10,1,726706606e0a49dbb6df3c3d51dc4ff8,0.0,0.000000,0.014956,0.0,0.194240,0.059482,...,0.513781,7.368982e-16,0.696882,0.519719,0.481555,0.500111,1.082585,0.200781,7.691412e-16,0.881804
4348,0,10,1,097fcd0bb39f4d60b9e22620ff30736a,0.0,0.000000,0.441322,0.0,0.019746,0.059482,...,0.495929,8.838310e-16,0.693245,0.520258,0.479412,0.500001,1.658828,0.110377,1.177329e-15,1.548452


In [12]:

# 构造 batches 并查看形状

ep = Episode(models=models, optimizers={}, config=Config, hyperparams=hp, device=device, episode_id=1)

sdf_batches = ep._create_firm_batches_from_df(df_firm, batch_size=batch_size, n_branches=n_branches)
pv_batches = ep._create_firm_batches_from_df(df_firm, batch_size=batch_size, n_branches=n_branches)
fc2_batches = ep._create_fc2_batches(df_firm, batch_size=batch_size, n_branches=n_branches)

print(f"batches -> sdf_fc1: {len(sdf_batches)}, policy_value: {len(pv_batches)}, fc2: {len(fc2_batches)}")

if sdf_batches:
    b0 = sdf_batches[0]
    print("sdf batch tensors:", {k: v.shape for k, v in b0.items() if hasattr(v, 'shape')})

if pv_batches:
    b0 = pv_batches[0]
    print("policy batch tensors:", {k: v.shape for k, v in b0.items() if hasattr(v, 'shape')})

if fc2_batches:
    b0 = fc2_batches[0]
    print("fc2 parent_states len:", len(b0['parent_states']), "children len:", len(b0['children_states']))
    if b0['parent_states']:
        print("fc2 parent shape:", b0['parent_states'][0].shape)
        print("fc2 child[0] shape:", b0['children_states'][0][0].shape)


batches -> sdf_fc1: 544, policy_value: 544, fc2: 4
sdf batch tensors: {'parent': torch.Size([8, 8]), 'child0': torch.Size([8, 8]), 'child1': torch.Size([8, 8])}
policy batch tensors: {'parent': torch.Size([8, 8]), 'child0': torch.Size([8, 8]), 'child1': torch.Size([8, 8])}
fc2 parent_states len: 8 children len: 8
fc2 parent shape: torch.Size([190, 7])
fc2 child[0] shape: torch.Size([190, 7])


In [14]:
pv_batches[0]

{'parent': tensor([[ 3.9786e-01,  5.3715e-02,  0.0000e+00,  1.3329e-01,  3.1397e-02,
          -1.0046e-01,  5.1213e+00,  3.4807e-25],
         [ 5.1576e-01, -1.4733e-01,  0.0000e+00,  1.5172e-01, -7.6897e-02,
          -2.0491e-01,  4.7852e+00,  2.4962e-27],
         [ 0.0000e+00,  5.7576e-02,  0.0000e+00,  8.1905e-02,  4.0208e-02,
          -1.1745e-01,  4.9061e+00,  4.6714e-25],
         [ 2.8999e-01, -1.6337e-01,  0.0000e+00,  1.3224e-01, -3.5095e-02,
          -1.1918e-01,  4.9500e+00,  1.7773e-26],
         [ 6.1081e-01, -6.4403e-02,  0.0000e+00,  9.9629e-02, -4.5214e-03,
          -1.1618e-01,  5.1576e+00,  2.0605e-26],
         [ 6.0716e-01, -1.8083e-01,  0.0000e+00,  6.0309e-02,  5.3665e-02,
          -2.3484e-02,  5.2705e+00,  1.0993e-22],
         [ 3.4347e-01, -2.5362e-01,  0.0000e+00,  1.1256e-01,  4.0208e-02,
          -1.1745e-01,  4.9061e+00,  4.6714e-25],
         [ 5.2835e-01, -1.3456e-01,  0.0000e+00,  1.1197e-01,  3.1688e-02,
          -1.2258e-01,  5.0543e+00,  3.0


## 校验 parent/child 对齐关系
基于 `_create_firm_batches_from_df` 生成的 batch，检查每个 parent 的 ID/path 是否与 children 对应。


In [17]:

# 校验 parent-children 对齐
# 使用已有的 df_firm/df_macro 和构建好的 batches（前面单元需先运行）

def check_alignment(df_firm, batch, n_branches=Config.BRANCH_NUM):
    # 只取前几个样本检查
    parent_t = batch['parent']
    for idx in range(min(200, parent_t.shape[0])):
        parent_row = df_firm.iloc[idx]
        path = parent_row['path']
        ID = parent_row['ID']
        t_parent = parent_row['t'] if isinstance(parent_row['t'], str) else int(parent_row['t'])
        print(f"Parent sample {idx}: path={path}, ID={ID}, t={t_parent}")
        for k in range(n_branches):
            if df_firm['t'].dtype == object:
                key_t = f't+1_{k}'
                child_row = df_firm[(df_firm['path']==path) & (df_firm['ID']==ID) & (df_firm['t']==key_t)]
            else:
                key_t = t_parent + 1
                child_row = df_firm[(df_firm['path']==path) & (df_firm['ID']==ID) & (df_firm['t']==key_t) & (df_firm['branch']==k)]
            if child_row.empty:
                print(f"  branch {k}: missing child")
            else:
                print(f"  branch {k}: matched rows={len(child_row)}")

if pv_batches:
    print("Checking alignment on first policy_value batch:")
    check_alignment(df_firm, pv_batches[0], n_branches=n_branches)
else:
    print("pv_batches is empty; run previous cells first.")


Checking alignment on first policy_value batch:
Parent sample 0: path=0, ID=6c735502c0ab4839b5f630b9f5dd7909, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 1: path=0, ID=1012c97ea81245db914f5d6df9ec3f69, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 2: path=0, ID=d820697891ec4cd7b9d0395c098e1be4, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 3: path=0, ID=67ae1926f8a94333929cbc3b74c87c11, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 4: path=0, ID=abdc61e8877c45439ed4dfaaca154a76, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 5: path=0, ID=2d252ff7b71340169a6769573b04b626, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 6: path=0, ID=fab3348dee6d4119bd3725b91f6c70df, t=0
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent sample 7: path=0, ID=920e8cb728e544e08a974e8ea57ba82b, t=0
  branch 0: matched rows=1
  branch 1: matched

In [18]:
df_firm

,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_i,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C
0,0,0,-1,6c735502c0ab4839b5f630b9f5dd7909,1.0,0.370202,0.090160,1.0,0.126619,-0.013420,...,0.349796,4.665106e-17,0.752077,0.552019,0.484954,0.528560,1.592163,0.212764,7.153271e-17,1.379399
1,0,0,-1,1012c97ea81245db914f5d6df9ec3f69,1.0,0.003481,-0.310053,0.0,0.193134,-0.013420,...,0.354465,5.421784e-17,0.749070,0.554078,0.476500,0.526579,0.908869,0.211582,5.868682e-17,0.697287
2,0,0,-1,d820697891ec4cd7b9d0395c098e1be4,1.0,0.261684,-0.553397,0.0,0.141070,-0.013420,...,0.365960,6.926503e-17,0.744172,0.554633,0.472643,0.524628,0.470990,0.125878,4.506307e-17,0.345111
3,0,0,-1,67ae1926f8a94333929cbc3b74c87c11,1.0,0.225235,-0.294209,0.0,0.029609,-0.013420,...,0.358588,6.910140e-17,0.744219,0.552662,0.470195,0.523090,0.392548,0.059063,3.201088e-17,0.333485
4,0,0,-1,abdc61e8877c45439ed4dfaaca154a76,1.0,0.056402,0.016489,0.0,0.078568,-0.013420,...,0.347262,4.888019e-17,0.751143,0.551830,0.473917,0.524774,1.389766,0.176352,6.782794e-17,1.213413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13045,2,10,1,e90c63fd674a414fa028a36f8d33178e,0.0,0.000000,0.166991,0.0,0.076393,0.004494,...,0.504612,9.246583e-16,0.692342,0.520228,0.481102,0.500484,1.193276,0.139274,1.016434e-15,1.054002
13046,2,10,1,a86e4a7bf1a543ab9383a4e40a7a9ddb,0.0,0.000000,-0.035949,0.0,0.183627,0.004494,...,0.498396,6.747033e-16,0.698645,0.521029,0.481589,0.501372,0.973861,0.192473,6.675659e-16,0.781389
13047,2,10,1,bd44406381bb49e4a04d8d8218225f31,0.0,0.000000,-0.387832,0.0,0.011124,0.004494,...,0.453099,3.757210e-16,0.710354,0.523831,0.483706,0.505651,0.684696,0.105520,3.173461e-16,0.579176
13048,2,10,1,a47b26d72dbe4e67a9258fbfa3f030fc,0.0,0.000000,0.061033,0.0,0.041536,0.004494,...,0.493582,8.483815e-16,0.694064,0.519887,0.483193,0.501776,1.073319,0.121133,8.817068e-16,0.952186


In [19]:

# 进一步查看 t>0 的 parent/child 对齐（simulate 模式适用）
if df_firm['t'].dtype != object:
    parents_t = df_firm[(df_firm['branch'] == -1) & (df_firm['t'] > 0)]
    if parents_t.empty:
        print("No parent rows with t>0 found.")
    else:
        print(f"parents with t>0: {len(parents_t)} (show up to 3)")
        for _, prow in parents_t.head(3).iterrows():
            path = prow['path']
            ID = prow['ID']
            t_parent = int(prow['t'])
            print(f"Parent path={path}, ID={ID}, t={t_parent}")
            for k in range(n_branches):
                child_row = df_firm[(df_firm['path']==path) & (df_firm['ID']==ID) & (df_firm['t']==t_parent+1) & (df_firm['branch']==k)]
                if child_row.empty:
                    print(f"  branch {k}: missing child")
                else:
                    print(f"  branch {k}: matched rows={len(child_row)}")
else:
    print("t is string type (sample模式)，仅有 t 和 t+1_k 行，无 t>0 的多期父节点可检查。")


parents with t>0: 4050 (show up to 3)
Parent path=0, ID=6c735502c0ab4839b5f630b9f5dd7909, t=1
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent path=0, ID=1012c97ea81245db914f5d6df9ec3f69, t=1
  branch 0: matched rows=1
  branch 1: matched rows=1
Parent path=0, ID=d820697891ec4cd7b9d0395c098e1be4, t=1
  branch 0: matched rows=1
  branch 1: matched rows=1
